# 🎸 Ukulele Tablature Generator
**Physics-aware polyphonic ukulele tab generation from raw audio.**

Pipeline:
1. FFT-based brute-force harmonic template matching → top-N note multiset candidates
2. Differentiable Karplus-Strong optimizer → fits physical parameters (decay, inharmonicity) per candidate
3. Combined spectral + physics score → best chord shape wins

> **GPU toggle:** Set `USE_GPU = True` in the *Device Configuration* cell below to run on Kaggle GPU.


## 1 · Install dependencies

In [ ]:
%%capture
!pip install librosa soundfile tqdm

## 2 · Clone / mount the project

Choose **one** of the cells below:
- **Option A (Kaggle Dataset):** Upload the repo as a Kaggle dataset and mount it.
- **Option B (GitHub):** Clone directly from GitHub (requires internet ON in notebook settings).

In [ ]:
# Option B — GitHub clone (enable Internet in Kaggle settings first)
import os
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # ← replace
if not os.path.exists("ukulele_tabs"):
    !git clone {REPO_URL} ukulele_tabs
os.chdir("ukulele_tabs/prototype")
print("Working directory:", os.getcwd())

## 3 · Device Configuration ⚙️

**Change `USE_GPU` here — this is the only place you need to touch.**

In [ ]:
# ─────────────────────────────────────────────
USE_GPU = True   # ← True = GPU (Kaggle T4/P100), False = CPU
# ─────────────────────────────────────────────

# Patch device_config at runtime so all modules pick it up
import device_config
import torch

if USE_GPU and torch.cuda.is_available():
    device_config.DEVICE = torch.device("cuda")
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    device_config.DEVICE = torch.device("cpu")
    if USE_GPU:
        print("⚠️  USE_GPU=True but no CUDA found — using CPU")
    else:
        print("ℹ️  Using CPU")

DEVICE = device_config.DEVICE
print(f"Device: {DEVICE}")

## 4 · Upload your audio file

In [ ]:
from google.colab import files  # remove this line if running on Kaggle
# On Kaggle: add your .wav as a dataset attachment instead

# uploaded = files.upload()  # Colab only
# AUDIO_FILE = list(uploaded.keys())[0]

# Kaggle: point to your input file
AUDIO_FILE = "/kaggle/input/YOUR_DATASET/your_chord.wav"   # ← replace
print(f"Audio: {AUDIO_FILE}")

## 5 · Configure the run

In [ ]:
# ── Pipeline parameters ────────────────────────────────────────────────────
STRUMMED_STRINGS = [1, 2, 3, 4]   # which strings were strummed (1=a, 2=e, 3=c, 4=g)
NUM_NOTES        = 4               # number of simultaneous notes
NUM_ITERATIONS   = 50              # optimizer iterations per candidate (50 = good balance)
LR               = 0.5             # Adam learning rate
TOP_N            = 16              # number of spectral candidates to evaluate
ALPHA            = 0.4             # spectral score weight  }
BETA             = 0.6             # physics score weight   } must sum to 1
# ──────────────────────────────────────────────────────────────────────────

## 6 · Run the pipeline

In [ ]:
import sys, os
import librosa
import numpy as np
import torch

from pitch_estimation import estimate_chord_shape, _extract_peak_energy_window
from optimization import optimize_multi_parameters
from fretboard_mapper import refine_chord_with_inharmonicity

_NOTE_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

def midi_to_name(m):
    return f"{_NOTE_NAMES[m % 12]}{m // 12 - 1}"

# ── Load audio ─────────────────────────────────────────────────────────────
waveform_np, sample_rate = librosa.load(AUDIO_FILE, sr=16000, mono=True)
waveform_full = torch.from_numpy(waveform_np).unsqueeze(0).to(DEVICE)
print(f"Loaded: {len(waveform_np)/sample_rate:.2f}s  @{sample_rate}Hz")

# ── Spectral candidate search ───────────────────────────────────────────────
print(f"\n[1] Spectral search → top {TOP_N} multisets ...")
chord_result = estimate_chord_shape(
    waveform_full, sample_rate, STRUMMED_STRINGS,
    top_n_multisets=TOP_N
)
candidates = chord_result["candidates"]

# ── Prepare onset window for optimizer ─────────────────────────────────────
_MAX_OPT_SEC = 0.5
y_opt = _extract_peak_energy_window(waveform_np, sample_rate, window_sec=_MAX_OPT_SEC)
y_opt = y_opt[:int(_MAX_OPT_SEC * sample_rate)]
waveform_opt = torch.from_numpy(y_opt).unsqueeze(0)

# ── Evaluate each candidate ─────────────────────────────────────────────────
print(f"\n[2] Evaluating {len(candidates)} candidates with physics optimizer ...")
evaluated = []

for rank, candidate in enumerate(candidates, 1):
    pitches   = list(candidate["pitches"])
    notes_str = ', '.join(midi_to_name(m) for m in candidate["multiset_midis"])
    print(f"\n  Candidate #{rank}: [{notes_str}]  spectral={candidate['spectral_score_norm']:.3f}")

    cand_pitches = pitches[:]
    active_mask  = [True] * len(STRUMMED_STRINGS)
    while len(cand_pitches) < NUM_NOTES:
        cand_pitches.append(440.0)
        active_mask.append(False)

    voice_params = optimize_multi_parameters(
        target_audio=waveform_opt,
        pitches=cand_pitches,
        sample_rate=sample_rate,
        num_iterations=NUM_ITERATIONS,
        lr=LR,
        active_mask=active_mask
    )

    active_voice_params = [voice_params[i] for i, a in enumerate(active_mask) if a]
    refined = refine_chord_with_inharmonicity(candidate, active_voice_params, STRUMMED_STRINGS)

    max_physics  = float(len(STRUMMED_STRINGS))
    raw_physics  = refined.get("inharm_score", 0.0)
    physics_norm = raw_physics / max_physics if max_physics > 0 else 0.0
    combined     = ALPHA * candidate["spectral_score_norm"] + BETA * physics_norm

    candidate.update({
        "voice_params"      : active_voice_params,
        "refined"           : refined,
        "physics_score"     : raw_physics,
        "physics_score_norm": physics_norm,
        "combined_score"    : combined,
    })
    evaluated.append(candidate)
    print(f"    → spectral={candidate['spectral_score_norm']:.3f}  "
          f"physics={physics_norm:.3f}  combined={combined:.3f}")

evaluated.sort(key=lambda c: -c["combined_score"])
winner = evaluated[0]

## 7 · Final Rankings & Tab

In [ ]:
STRING_NAMES = {1: 'a', 2: 'e', 3: 'c', 4: 'g'}

print("\n" + "─"*62)
print("  Candidate Rankings (spectral + physics):\n")
for i, c in enumerate(evaluated, 1):
    notes  = ', '.join(midi_to_name(m) for m in c["multiset_midis"])
    marker = "  ◄── WINNER" if i == 1 else ""
    print(f"  #{i:>2}: [{notes}]")
    print(f"       spectral={c['spectral_score_norm']:.3f}  "
          f"physics={c['physics_score_norm']:.3f}  "
          f"combined={c['combined_score']:.3f}{marker}")

print("\n" + "─"*62)
print("  Final Tablature:\n")
mapping = winner["refined"]["mapping"]
for s_idx in sorted(STRUMMED_STRINGS, reverse=True):
    fret = mapping[s_idx]
    print(f"  {STRING_NAMES[s_idx]}:{fret}  ", end="")
print()

if "inharmonicity_scores" in winner["refined"]:
    print("\n  Inharmonicity diagnostics:")
    for s_idx in sorted(STRUMMED_STRINGS, reverse=True):
        d = winner["refined"]["inharmonicity_scores"][s_idx]
        print(f"  {STRING_NAMES[s_idx]}:{d['fret']}  "
              f"obs={d['observed_stretch']:.4f}  "
              f"exp={d['expected_stretch']:.4f}  "
              f"score={d['score']:.3f}")